In [1]:
import pandas as pd 
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
from sklearn.metrics import  classification_report
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from sklearn.metrics import precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error,r2_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection  import GridSearchCV

print("Libararies Exported")

Libararies Exported


Follwed same step as spliting and scaling from part 2 as per Task requiremnet

In [2]:
df=pd.read_csv("../Data/cleaned_data.csv")
print("Loaded csv file successfully!")
print(f"First 5 rows:\n{df.head(5)}")
print(f"Datatypes:\n{df.dtypes}")
print(f"Shape of Dataframe:\n{df.shape}")
print(f"Null value counts:\n{df.isnull().sum()}")
print(f"Null value percentage:\n{round((df.isnull().sum() / df.shape[0] )* 100,2)}")

risk_mapping = {
    'Good': 0,
    'Satisfactory': 0,
    'Moderate': 0,
    'Poor': 1,
    'Very Poor': 1,
    'Severe': 1
}
# df["AQI_Bucket"].unique()

  
    
X=df.drop(columns=['Date','AQI','AQI_Bucket'])
y_reg=df["AQI"]
y_clf=df["AQI_Bucket"].map(risk_mapping)

print("Feature and Label classifications are done and definitions provided in README file ")
# print(X)
# print(y_reg)
# print(y_clf)
def enso_category(oni):
    if oni>=0.5:
        return "EL Nino"
    elif oni <=-0.5:
        return "La Nina"
    else: 
        return "Neutral"
X["ENSO_Category"]=X["ONI"].apply(enso_category)

X=pd.get_dummies(X,columns=['City','season','ENSO_Category'],drop_first=True)
print("Categorical Feature encoding completed!")
X_train_reg,X_test_reg,y_train_reg,y_test_reg=train_test_split(X,y_reg,test_size=0.2,random_state=42)
X_train_clf,X_test_clf,y_train_clf,y_test_clf=train_test_split(X,y_clf,test_size=0.2,random_state=42)

scaler_reg=StandardScaler()
X_train_reg_scaled=scaler_reg.fit_transform(X_train_reg)
X_test_reg_scaled=scaler_reg.transform(X_test_reg)

scaler_clf=StandardScaler()
X_train_clf_scaled=scaler_clf.fit_transform(X_train_clf)
X_test_clf_scaled=scaler_clf.transform(X_test_clf)

print("Training and testings set are splitted and scaled accrodingly")




Loaded csv file successfully!
First 5 rows:
        City        Date  season AQI_Bucket  avg_temp  min_temp  max_temp  \
0  Ahmedabad  2018-01-01  Winter       Poor      20.3      10.8      27.2   
1  Ahmedabad  2018-01-02  Winter       Poor      19.8      12.2      26.0   
2  Ahmedabad  2018-01-03  Winter       Poor      19.5      12.5      26.0   
3  Ahmedabad  2018-01-04  Winter     Severe      19.6      10.5      28.0   
4  Ahmedabad  2018-01-05  Winter     Severe      19.5       8.6      27.0   

   wind_speed  air_pressure    AQI   ONI  
0         5.4        1011.8  278.0 -0.77  
1         8.5        1012.5  256.0 -0.77  
2         8.3        1013.3  300.0 -0.77  
3         4.1        1013.7  532.0 -0.77  
4         3.0        1012.8  534.0 -0.77  
Datatypes:
City             object
Date             object
season           object
AQI_Bucket       object
avg_temp        float64
min_temp        float64
max_temp        float64
wind_speed      float64
air_pressure    float64
AQI     

Task1 _decision Tree Baseline

In [3]:
decisiontree=DecisionTreeClassifier(random_state=42)
decisiontree.fit(X_train_clf_scaled,y_train_clf)
y_train_pred=decisiontree.predict(X_train_clf_scaled)
y_test_pred=decisiontree.predict(X_test_clf_scaled)

train_accuracy = accuracy_score(y_train_clf, y_train_pred)
test_accuracy = accuracy_score(y_test_clf, y_test_pred)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

Training Accuracy: 0.9999
Test Accuracy: 0.8902


Task 2 Controlled Decision Tree

In [4]:
controlled_tree = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    random_state=42
)

controlled_tree.fit(X_train_clf_scaled, y_train_clf)

y_train_pred_cntl = controlled_tree.predict(X_train_clf_scaled)
y_test_pred_cntl = controlled_tree.predict(X_test_clf_scaled)

train_accuracy_cntl = accuracy_score(y_train_clf, y_train_pred_cntl)
test_accuracy_cntl = accuracy_score(y_test_clf, y_test_pred_cntl)

print(f" Controlled Training Accuracy: {train_accuracy_cntl:.4f}")
print(f" Controlled Test Accuracy: {test_accuracy_cntl:.4f}")

 Controlled Training Accuracy: 0.8743
 Controlled Test Accuracy: 0.8772


Task 3 Gini vs Entropy

In [5]:
gini_tree = DecisionTreeClassifier(
    criterion='gini',
    max_depth=5,
    random_state=42
)

entropy_tree = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=5,
    random_state=42
)


gini_tree.fit(X_train_clf_scaled, y_train_clf)
entropy_tree.fit(X_train_clf_scaled, y_train_clf)


gini_pred = gini_tree.predict(X_test_clf_scaled)
entropy_pred = entropy_tree.predict(X_test_clf_scaled)


gini_accuracy = accuracy_score(y_test_clf, gini_pred)
entropy_accuracy = accuracy_score(y_test_clf, entropy_pred)

print(f"Gini Test Accuracy: {round(gini_accuracy,4)}")
print(f"Entropy Test Accuracy: {round(entropy_accuracy,4)}")

Gini Test Accuracy: 0.8772
Entropy Test Accuracy: 0.8605


Task 4 -Random Forest

In [6]:
random_forest = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)
random_forest.fit(X_train_clf_scaled, y_train_clf)
y_train_pred_rf = random_forest.predict(X_train_clf_scaled)
y_test_pred_rf = random_forest.predict(X_test_clf_scaled)
y_test_prob_rf = random_forest.predict_proba(X_test_clf_scaled)[:, 1]
train_accuracy_rf = accuracy_score(y_train_clf, y_train_pred_rf)
test_accuracy_rf = accuracy_score(y_test_clf, y_test_pred_rf)
roc_auc_rf = roc_auc_score(y_test_clf, y_test_prob_rf)


print(f"Training Accuracy: {round(train_accuracy_rf,4)}")
print(f"Test Accuracy: {round(test_accuracy_rf,4)}")
print(f"ROC-AUC: {round(roc_auc_rf,4)}")

feature_imp = pd.DataFrame({
    "Feature": X_train_clf.columns,
    "Importance": random_forest.feature_importances_
})

top5_features = feature_imp.sort_values(
    by="Importance",
    ascending=False
).head(5)

print("\nTop 5 Feature Importances:")
print(top5_features)

Training Accuracy: 0.9187
Test Accuracy: 0.8943
ROC-AUC: 0.9408

Top 5 Feature Importances:
        Feature  Importance
1      min_temp    0.144954
3    wind_speed    0.124637
2      max_temp    0.112303
0      avg_temp    0.108318
4  air_pressure    0.087741


Task 4 a - Gradient Bossting

In [7]:
gradient_boosting = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gradient_boosting.fit(X_train_clf_scaled, y_train_clf)

# Predictions
y_train_pred_gb = gradient_boosting.predict(X_train_clf_scaled)
y_test_pred_gb = gradient_boosting.predict(X_test_clf_scaled)
y_test_prob_gb = gradient_boosting.predict_proba(X_test_clf_scaled)[:, 1]

# Metrics
train_accuracy_gb = accuracy_score(y_train_clf, y_train_pred_gb)
test_accuracy_gb = accuracy_score(y_test_clf, y_test_pred_gb)
roc_auc_gb = roc_auc_score(y_test_clf, y_test_prob_gb)

print(f"Training Accuracy: {train_accuracy_gb:.4f}")
print(f"Test Accuracy: {test_accuracy_gb:.4f}")
print(f"ROC-AUC: {roc_auc_gb:.4f}")

Training Accuracy: 0.8975
Test Accuracy: 0.8970
ROC-AUC: 0.9411


Task 4 b - Feature abalation

In [8]:

lowest_features = feature_imp.sort_values(
    by="Importance",
    ascending=True
).head(5)["Feature"].tolist()

print("Lowest 5 Features:")
print(lowest_features)


X_train_reduced = X_train_clf.drop(columns=lowest_features)
X_test_reduced = X_test_clf.drop(columns=lowest_features)


reduced_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

reduced_rf.fit(X_train_reduced, y_train_clf)


reduced_prob = reduced_rf.predict_proba(X_test_reduced)[:, 1]
reduced_auc = roc_auc_score(y_test_clf, reduced_prob)

print(f"\nFull Model ROC-AUC: {round(roc_auc_rf,4)}")
print(f"Reduced Model ROC-AUC: {round(reduced_auc,4)}")

Lowest 5 Features:
['City_Ernakulam', 'City_Bhopal', 'City_Coimbatore', 'City_Chandigarh', 'City_Mumbai']

Full Model ROC-AUC: 0.9408
Reduced Model ROC-AUC: 0.9388


In [9]:
print(type(X_train_clf))
print(type(X_train_clf_scaled))

<class 'pandas.core.frame.DataFrame'>
<class 'numpy.ndarray'>


Task 5 -Cross Validaion

In [10]:



cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Create each model
logistic_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        C=1.0,
        random_state=42,
        max_iter=1000
    )
)

decision_tree_model = make_pipeline(
    StandardScaler(),
    DecisionTreeClassifier(
        max_depth=5,
        min_samples_split=20,
        random_state=42
    )
)

random_forest_model = make_pipeline(
    StandardScaler(),
    RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42
    )
)

gradient_boosting_model = make_pipeline(
    StandardScaler(),
    GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
)


models = {
    "Logistic Regression": logistic_model,
    "Controlled Decision Tree": decision_tree_model,
    "Random Forest": random_forest_model,
    "Gradient Boosting": gradient_boosting_model
}


results = []


for model_name, model in models.items():

    auc_scores = cross_val_score(
        estimator=model,
        X=X_train_clf,
        y=y_train_clf,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )

    mean_auc = auc_scores.mean()
    std_auc = auc_scores.std()

    results.append({
        "Model": model_name,
        "Mean AUC": mean_auc,
        "Std AUC": std_auc
    })


cv_results = pd.DataFrame(results)

print(cv_results)

                      Model  Mean AUC   Std AUC
0       Logistic Regression  0.943248  0.005644
1  Controlled Decision Tree  0.834038  0.012608
2             Random Forest  0.926723  0.010624
3         Gradient Boosting  0.929707  0.007983


Task 6 - Hyper parameter Tuning 

In [11]:
pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    RandomForestClassifier(random_state=42)
)
param_grid = {
    "randomforestclassifier__n_estimators": [50, 100, 200],
    "randomforestclassifier__max_depth": [5, 10, None],
    "randomforestclassifier__min_samples_leaf": [1, 5]
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)


grid_search.fit(X_train_clf, y_train_clf)


print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validated ROC-AUC:")
print(f"{grid_search.best_score_:.4f}")

Best Parameters:
{'randomforestclassifier__max_depth': None, 'randomforestclassifier__min_samples_leaf': 1, 'randomforestclassifier__n_estimators': 200}

Best Cross-Validated ROC-AUC:
0.9587


Task 7 Manual learning curve

Task 8 -Serialize  the bets model 